# 🎓 Fine-Tuning Qwen3.5-4B con LoRA per ChatBDI
Notebook per l'addestramento del modello con QLoRA su Google Colab (GPU T4).

In [ ]:
# ==========================================
# 0. INSTALLAZIONE DIPENDENZE (solo prima volta)
# ==========================================
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps trl peft accelerate bitsandbytes datasets -q

In [ ]:
import torch
import gc
from google.colab import drive
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig

# ==========================================
# 1. COLLEGAMENTO A GOOGLE DRIVE
# ==========================================
drive.mount('/content/drive')

# Pulizia preventiva VRAM
gc.collect()
torch.cuda.empty_cache()
print("Librerie importate e GPU pronta!")

## 📂 Caricamento Dataset
Carica il dataset dal Drive e lo divide in 90% Training / 10% Validation.

In [ ]:
# ==========================================
# 2. CARICAMENTO E SPLITTING DEL DATASET
# ==========================================
print("Caricamento del dataset...")
dataset_path = "/content/drive/MyDrive/Tirocinio_bechelor/dataset_short_R2.jsonl"
full_dataset = load_dataset("json", data_files=dataset_path, split="train")

# Split 90% Training / 10% Validation
dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['test'])}")

## 🧠 Caricamento Modello e Configurazione LoRA
Carica Qwen3.5-4B in 4-bit e configura gli adattatori LoRA.

In [ ]:
# ==========================================
# 3. CARICAMENTO MODELLO E QLORA
# ==========================================
print("Caricamento del modello Qwen3.5-4B...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-4B",
    max_seq_length = 512,   # Ottimizzato in base alla lunghezza dei nostri dati
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,          # Alpha = 2 * Rank
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_dora = False,
)
print("Modello e adattatori LoRA pronti!")

## 🏋️ Formattazione ChatML e Addestramento
Formatta i dati nel template ChatML ed esegue il fine-tuning supervisionato.

In [ ]:
# ==========================================
# 4. FORMATTAZIONE CHATML
# ==========================================
print("Formattazione dei dati in ChatML...")
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

def formatta_dati(esempi):
    testi = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in esempi["messages"]]
    return {"text": testi}

dataset = dataset.map(formatta_dati, batched=True)
print("Dati formattati!")

# ==========================================
# 5. ADDESTRAMENTO
# ==========================================
print("Inizio addestramento...")

gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
    dataset_text_field = "text",
    max_seq_length = 512,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 4,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/Tirocinio_bechelor/risultati_chatbdi_checkpoints_qwen3.5-4B",
    ),
)

trainer.train()
print("Addestramento completato!")

## 💾 Salvataggio Adattatori e Modello
Salva gli adattatori LoRA su Google Drive ed esporta il modello in formato GGUF.

In [ ]:
# ==========================================
# 6. SALVATAGGIO ADATTATORI LORA
# ==========================================
print("Salvataggio adattatori LoRA su Google Drive...")
lora_path = "/content/drive/MyDrive/Tirocinio_bechelor/lora_chatbdi_qwen3.5-4B"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"Adattatori LoRA salvati in: {lora_path}")

# ==========================================
# 7. ESPORTAZIONE GGUF
# ==========================================
print("Esportazione in GGUF su Google Drive (ci vorranno alcuni minuti)...")
export_path = "/content/drive/MyDrive/Tirocinio_bechelor/modello_chatbdi_qwen3.5-4B"
model.save_pretrained_gguf(
    export_path,
    tokenizer,
    quantization_method = "q4_k_m",
)
print(f"\N{PARTY POPPER} FINITO! Modello GGUF salvato in: {export_path}")

## 🧪 Test Rapido del Modello Fine-Tuned
Verifica rapida su 21 campioni di test per controllare che il modello produca JSON corretti.

In [ ]:
import json
import re
import torch
from transformers import StoppingCriteria, StoppingCriteriaList

# ==========================================
# 8. TEST RAPIDO SU CAMPIONI DI VERIFICA
# ==========================================
DRIVE_BASE = "/content/drive/MyDrive/Tirocinio_bechelor"
TEST_DATA = f"{DRIVE_BASE}/tickets_test_dataset.jsonl"

def normalize(obj):
    if not isinstance(obj, dict):
        return obj
    out = {}
    for k, v in obj.items():
        if isinstance(v, str):
            v = v.strip()
            if v == "_" or (v and v[0].isupper() and v.replace("_", "").isalpha()):
                v = "__VAR__"
        out[k] = v
    return out

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = set([s for s in stop_ids if s is not None])
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1].item() in self.stop_ids

# Prepara il modello per l'inferenza
FastLanguageModel.for_inference(model)
text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
im_end_id = text_tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_crit = StoppingCriteriaList([StopOnTokens([text_tokenizer.eos_token_id, im_end_id])])

with open(TEST_DATA, encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f if line.strip()]

print(f"Test da eseguire: {len(test_cases)}\n")

passed_count = 0
for i, tc in enumerate(test_cases, 1):
    system = tc["messages"][0]["content"]
    user = tc["messages"][1]["content"]
    expected = tc["messages"][2]["content"]

    prompt_testuale = text_tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=False
    )
    prompt_testuale += "<|im_start|>assistant\n{"

    tokens = text_tokenizer(prompt_testuale, return_tensors="pt")
    input_ids = tokens["input_ids"].to("cuda")
    attention_mask = tokens["attention_mask"].to("cuda") if "attention_mask" in tokens else torch.ones_like(input_ids)
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=text_tokenizer.eos_token_id,
            stopping_criteria=stop_crit,
        )

    predicted_raw = text_tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True).strip()
    predicted_raw = re.sub(r'<think>.*?</think>', '', predicted_raw, flags=re.DOTALL).strip()
    predicted_raw = "{" + predicted_raw

    cleaned = predicted_raw
    backticks = "`" * 3
    if cleaned.startswith(backticks + "json"): cleaned = cleaned[7:]
    elif cleaned.startswith(backticks): cleaned = cleaned[3:]
    if cleaned.endswith(backticks): cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        passed = normalize(json.loads(cleaned)) == normalize(json.loads(expected))
    except json.JSONDecodeError:
        passed = False

    if passed:
        passed_count += 1

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] [{i:2d}] {cleaned}")
    if not passed:
        print(f"       Atteso: {expected}")

print(f"\nRisultato: {passed_count}/{len(test_cases)} test superati")